# MIT Mini Cheetah — RL Locomotion (Colab)

**Quadruped locomotion with PPO + MuJoCo**

This notebook is fully self-contained. Just run all cells top-to-bottom:
1. Install dependencies
2. Write source files to disk
3. Train PPO policy (MLP [512, 256, 128])
4. Evaluate and visualize results

> Runtime → Change runtime type → **T4 GPU** recommended (training uses CPU for MLP, but GPU helps PyTorch ops)


In [ ]:
!pip install -q gymnasium stable-baselines3[extra] mujoco tensorboard matplotlib scipy trimesh


## 1. Project Setup
Create directory structure and write all source files.

In [ ]:
import os
for d in [
    "assets",
    "src", "src/env", "src/training", "src/control", "src/robot",
    "src/utils", "src/visualization",
    "scripts",
    "logs/training", "logs/training/eval",
    "checkpoints/best",
]:
    os.makedirs(d, exist_ok=True)

# Create empty __init__.py files
for d in ["src", "src/env", "src/training", "src/control",
           "src/robot", "src/utils", "src/visualization"]:
    open(os.path.join(d, "__init__.py"), "w").close()

print("Directory structure created.")

In [ ]:
# Download meshes and URDF from Mini-Cheetah-ROS repo, then build the MJCF model.
# All kinematics/inertia come from the real URDF; meshes are the original STL files.
# Source: https://github.com/sevocrear/Mini-Cheetah-ROS

import pathlib, urllib.request, time, trimesh
import numpy as np

os.makedirs("assets/meshes", exist_ok=True)

BASE_MESH_URL = "https://raw.githubusercontent.com/sevocrear/Mini-Cheetah-ROS/master/cheetah_description/meshes/"
MESH_FILES = ["trunk.STL", "hip.STL", "thigh.STL", "calf.STL", "foot.STL"]
MAX_FACES = 100_000  # MuJoCo hard limit is 200K

def _download(url, dest, retries=5):
    for attempt in range(retries):
        try:
            urllib.request.urlretrieve(url, dest)
            return
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2)
            else:
                raise

print("Downloading STL meshes...")
for fname in MESH_FILES:
    dest = f"assets/meshes/{fname}"
    _download(BASE_MESH_URL + fname, dest)
    # Decimate if over MuJoCo face limit and re-export as proper binary STL
    mesh = trimesh.load(dest)
    nf = len(mesh.faces)
    if nf > MAX_FACES:
        try:
            mesh = mesh.simplify_quadric_decimation(MAX_FACES)
        except Exception:
            idx = np.random.choice(nf, MAX_FACES, replace=False)
            mesh = trimesh.Trimesh(vertices=mesh.vertices, faces=mesh.faces[idx])
    mesh.export(dest, file_type="stl")
    print(f"  {fname}: {len(mesh.faces)} faces")

print("Writing assets/mini_cheetah.xml (MJCF built from URDF kinematics + STL meshes)...")
pathlib.Path("assets/mini_cheetah.xml").write_text("""\
<?xml version="1.0" encoding="utf-8"?>
<!--
  MIT Mini Cheetah MuJoCo model
  Converted from URDF: https://github.com/sevocrear/Mini-Cheetah-ROS
  Meshes from: cheetah_description/meshes/
  Kinematics and inertia from the original URDF.
-->
<mujoco model="mini_cheetah">
  <compiler angle="radian" autolimits="true" meshdir="meshes"/>
  <option timestep="0.002" gravity="0 0 -9.81" iterations="50" solver="Newton" cone="elliptic"/>

  <visual>
    <headlight ambient="0.5 0.5 0.5" diffuse="0.8 0.8 0.8" specular="0.1 0.1 0.1"/>
    <map force="0.1" zfar="30"/>
    <quality shadowsize="2048"/>
  </visual>

  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.4 0.6 0.8" rgb2="0.05 0.05 0.2" width="100" height="100"/>
    <texture name="floor_tex" type="2d" builtin="checker" rgb1="0.6 0.6 0.6" rgb2="0.4 0.4 0.4" width="300" height="300"/>
    <material name="floor_mat" texture="floor_tex" texrepeat="10 10" specular="0.5" shininess="0.3" reflectance="0.1"/>
    <material name="body_mat" rgba="0.82 0.82 0.82 1" specular="0.5" shininess="0.8"/>
    <material name="hip_mat"  rgba="0.81 0.68 0.30 1" specular="0.3" shininess="0.5"/>
    <material name="leg_mat"  rgba="0.46 0.80 0.99 1" specular="0.3" shininess="0.5"/>
    <material name="foot_mat" rgba="0.49 0.49 0.49 1" specular="0.2" shininess="0.3"/>

    <mesh name="trunk_mesh" file="trunk.STL"/>
    <mesh name="hip_mesh"   file="hip.STL"/>
    <mesh name="thigh_mesh" file="thigh.STL"/>
    <mesh name="calf_mesh"  file="calf.STL"/>
    <mesh name="foot_mesh"  file="foot.STL"/>
  </asset>

  <default>
    <joint limited="true" armature="0.01" damping="0.5"/>
    <geom condim="6" friction="0.8 0.02 0.01" margin="0.001"/>
    <motor ctrllimited="true" ctrlrange="-20.0 20.0"/>
  </default>

  <worldbody>
    <light name="top" pos="0 0 3" dir="0 0 -1" directional="true" diffuse="0.8 0.8 0.8" specular="0.2 0.2 0.2"/>
    <light name="front" pos="2 -2 2" dir="-1 1 -1" diffuse="0.3 0.3 0.3"/>
    <geom name="floor" type="plane" size="50 50 0.1" material="floor_mat" condim="6" friction="1.0 0.005 0.0001"/>

    <body name="base" pos="0 0 0.35">
      <freejoint name="root"/>
      <inertial pos="0 0 0" mass="3.3" diaginertia="0.011253 0.036203 0.042673"/>
      <geom name="trunk_visual"    type="mesh" mesh="trunk_mesh" material="body_mat" contype="0" conaffinity="0"/>
      <geom name="trunk_collision" type="box"  size="0.15 0.1 0.0465"               contype="0" conaffinity="0"/>
      <site name="imu_site" pos="0 0 0" size="0.01"/>

      <!-- FR (Front-Right) -->
      <body name="FR_hip" pos="0.196 -0.049664 0">
        <inertial pos="0 0.036 0" quat="0.707107 0 0 0.707107" mass="0.57" diaginertia="0.000795577 0.000714827 0.00068875"/>
        <joint name="FR_hip_joint"   axis="1 0 0" range="-1.0472 0.872665"/>
        <geom name="FR_hip_visual"   type="mesh" mesh="hip_mesh"  material="hip_mat" euler="0 3.14159 0" contype="0" conaffinity="0"/>
        <geom name="FR_hip_col"      type="box"  size="0.045 0.04 0.0465"                               contype="0" conaffinity="0"/>
        <body name="FR_thigh" pos="0 -0.077476 0">
          <inertial pos="0 0.016 -0.11" quat="0.707107 0 0 0.707107" mass="0.634" diaginertia="0.00265482 0.00261821 0.000158764"/>
          <joint name="FR_thigh_joint" axis="0 1 0" range="-0.523599 3.92699"/>
          <geom name="FR_thigh_visual" type="mesh" mesh="thigh_mesh" material="leg_mat"              contype="0" conaffinity="0"/>
          <geom name="FR_thigh_col"    type="box"  size="0.0215 0.017 0.11" pos="0 0 -0.11"          contype="0" conaffinity="0"/>
          <body name="FR_calf" pos="0 0 -0.2115">
            <inertial pos="0 0 -0.161488" mass="0.214" diaginertia="0.00263797 0.00263797 4.48657e-05"/>
            <joint name="FR_calf_joint" axis="0 1 0" range="-3.14159 3.14159"/>
            <geom name="FR_calf_visual" type="mesh" mesh="calf_mesh" pos="0 0.009 0" material="leg_mat" contype="0" conaffinity="0"/>
            <geom name="FR_calf_col"    type="box"  size="0.008 0.008 0.1" pos="0 0 -0.1"               contype="0" conaffinity="0"/>
            <body name="FR_foot" pos="0 0 -0.23039">
              <inertial pos="0 0 0" mass="0.15" diaginertia="4.2135e-05 4.2135e-05 4.2135e-05"/>
              <geom name="FR_foot_visual" type="mesh"   mesh="foot_mesh" pos="0 0.018 0.02" material="foot_mat" contype="0" conaffinity="0"/>
              <geom name="FR_foot_col"    type="sphere" size="0.025"     pos="0 0 0.025"    material="foot_mat" condim="6" friction="1.5 0.005 0.0001" solimp="0.9 0.95 0.001" solref="0.02 1"/>
              <site name="FR_foot_site" pos="0 0 0" size="0.01"/>
            </body>
          </body>
        </body>
      </body>

      <!-- FL (Front-Left) -->
      <body name="FL_hip" pos="0.196 0.049664 0">
        <inertial pos="0 -0.036 0" quat="0.707107 0 0 0.707107" mass="0.57" diaginertia="0.000795577 0.000714827 0.00068875"/>
        <joint name="FL_hip_joint"   axis="1 0 0" range="-0.872665 1.0472"/>
        <geom name="FL_hip_visual"   type="mesh" mesh="hip_mesh"  material="hip_mat" euler="3.14159 3.14159 0" contype="0" conaffinity="0"/>
        <geom name="FL_hip_col"      type="box"  size="0.045 0.04 0.0465"                                      contype="0" conaffinity="0"/>
        <body name="FL_thigh" pos="0 0.077476 0">
          <inertial pos="0 -0.016 -0.11" quat="0.707107 0 0 0.707107" mass="0.634" diaginertia="0.00265482 0.00261821 0.000158764"/>
          <joint name="FL_thigh_joint" axis="0 1 0" range="-0.523599 3.92699"/>
          <geom name="FL_thigh_visual" type="mesh" mesh="thigh_mesh" material="leg_mat" euler="3.14159 3.14159 0" contype="0" conaffinity="0"/>
          <geom name="FL_thigh_col"    type="box"  size="0.0215 0.017 0.11" pos="0 0 -0.11"                    contype="0" conaffinity="0"/>
          <body name="FL_calf" pos="0 0 -0.2115">
            <inertial pos="0 0 -0.161488" mass="0.214" diaginertia="0.00263797 0.00263797 4.48657e-05"/>
            <joint name="FL_calf_joint" axis="0 1 0" range="-3.14159 3.14159"/>
            <geom name="FL_calf_visual" type="mesh" mesh="calf_mesh" pos="0 0.009 0" material="leg_mat" contype="0" conaffinity="0"/>
            <geom name="FL_calf_col"    type="box"  size="0.008 0.008 0.1" pos="0 0 -0.1"               contype="0" conaffinity="0"/>
            <body name="FL_foot" pos="0 0 -0.23039">
              <inertial pos="0 0 0" mass="0.15" diaginertia="4.2135e-05 4.2135e-05 4.2135e-05"/>
              <geom name="FL_foot_visual" type="mesh"   mesh="foot_mesh" pos="0 0.018 0.02" material="foot_mat" contype="0" conaffinity="0"/>
              <geom name="FL_foot_col"    type="sphere" size="0.025"     pos="0 0 0.025"    material="foot_mat" condim="6" friction="1.5 0.005 0.0001" solimp="0.9 0.95 0.001" solref="0.02 1"/>
              <site name="FL_foot_site" pos="0 0 0" size="0.01"/>
            </body>
          </body>
        </body>
      </body>

      <!-- RR (Rear-Right) -->
      <body name="RR_hip" pos="-0.196 -0.049664 0">
        <inertial pos="0 0.036 0" quat="0.707107 0 0 0.707107" mass="0.57" diaginertia="0.000795577 0.000714827 0.00068875"/>
        <joint name="RR_hip_joint"   axis="1 0 0" range="-1.0472 0.872665"/>
        <geom name="RR_hip_visual"   type="mesh" mesh="hip_mesh"  material="hip_mat" contype="0" conaffinity="0"/>
        <geom name="RR_hip_col"      type="box"  size="0.045 0.04 0.0465"            contype="0" conaffinity="0"/>
        <body name="RR_thigh" pos="0 -0.077476 0">
          <inertial pos="0 0.016 -0.11" quat="0.707107 0 0 0.707107" mass="0.634" diaginertia="0.00265482 0.00261821 0.000158764"/>
          <joint name="RR_thigh_joint" axis="0 1 0" range="-0.523599 3.92699"/>
          <geom name="RR_thigh_visual" type="mesh" mesh="thigh_mesh" material="leg_mat" contype="0" conaffinity="0"/>
          <geom name="RR_thigh_col"    type="box"  size="0.0215 0.017 0.11" pos="0 0 -0.11" contype="0" conaffinity="0"/>
          <body name="RR_calf" pos="0 0 -0.2115">
            <inertial pos="0 0 -0.161488" mass="0.214" diaginertia="0.00263797 0.00263797 4.48657e-05"/>
            <joint name="RR_calf_joint" axis="0 1 0" range="-3.14159 3.14159"/>
            <geom name="RR_calf_visual" type="mesh" mesh="calf_mesh" pos="0 0.009 0" material="leg_mat" contype="0" conaffinity="0"/>
            <geom name="RR_calf_col"    type="box"  size="0.008 0.008 0.1" pos="0 0 -0.1"   contype="0" conaffinity="0"/>
            <body name="RR_foot" pos="0 0 -0.23039">
              <inertial pos="0 0 0" mass="0.15" diaginertia="4.2135e-05 4.2135e-05 4.2135e-05"/>
              <geom name="RR_foot_visual" type="mesh"   mesh="foot_mesh" pos="0 0.018 0.02" material="foot_mat" contype="0" conaffinity="0"/>
              <geom name="RR_foot_col"    type="sphere" size="0.025"     pos="0 0 0.025"    material="foot_mat" condim="6" friction="1.5 0.005 0.0001" solimp="0.9 0.95 0.001" solref="0.02 1"/>
              <site name="RR_foot_site" pos="0 0 0" size="0.01"/>
            </body>
          </body>
        </body>
      </body>

      <!-- RL (Rear-Left) -->
      <body name="RL_hip" pos="-0.196 0.049664 0">
        <inertial pos="0 -0.036 0" quat="0.707107 0 0 0.707107" mass="0.57" diaginertia="0.000795577 0.000714827 0.00068875"/>
        <joint name="RL_hip_joint"   axis="1 0 0" range="-0.872665 1.0472"/>
        <geom name="RL_hip_visual"   type="mesh" mesh="hip_mesh"  material="hip_mat" euler="3.14159 0 0" contype="0" conaffinity="0"/>
        <geom name="RL_hip_col"      type="box"  size="0.045 0.04 0.0465"                               contype="0" conaffinity="0"/>
        <body name="RL_thigh" pos="0 0.077476 0">
          <inertial pos="0 -0.016 -0.11" quat="0.707107 0 0 0.707107" mass="0.634" diaginertia="0.00265482 0.00261821 0.000158764"/>
          <joint name="RL_thigh_joint" axis="0 1 0" range="-0.523599 3.92699"/>
          <geom name="RL_thigh_visual" type="mesh" mesh="thigh_mesh" material="leg_mat" euler="3.14159 3.14159 0" contype="0" conaffinity="0"/>
          <geom name="RL_thigh_col"    type="box"  size="0.0215 0.017 0.11" pos="0 0 -0.11"            contype="0" conaffinity="0"/>
          <body name="RL_calf" pos="0 0 -0.2115">
            <inertial pos="0 0 -0.161488" mass="0.214" diaginertia="0.00263797 0.00263797 4.48657e-05"/>
            <joint name="RL_calf_joint" axis="0 1 0" range="-3.14159 3.14159"/>
            <geom name="RL_calf_visual" type="mesh" mesh="calf_mesh" pos="0 0.009 0" material="leg_mat" contype="0" conaffinity="0"/>
            <geom name="RL_calf_col"    type="box"  size="0.008 0.008 0.1" pos="0 0 -0.1"               contype="0" conaffinity="0"/>
            <body name="RL_foot" pos="0 0 -0.23039">
              <inertial pos="0 0 0" mass="0.15" diaginertia="4.2135e-05 4.2135e-05 4.2135e-05"/>
              <geom name="RL_foot_visual" type="mesh"   mesh="foot_mesh" pos="0 0.018 0.02" material="foot_mat" contype="0" conaffinity="0"/>
              <geom name="RL_foot_col"    type="sphere" size="0.025"     pos="0 0 0.025"    material="foot_mat" condim="6" friction="1.5 0.005 0.0001" solimp="0.9 0.95 0.001" solref="0.02 1"/>
              <site name="RL_foot_site" pos="0 0 0" size="0.01"/>
            </body>
          </body>
        </body>
      </body>

    </body>
  </worldbody>

  <actuator>
    <motor name="FR_hip_motor"   joint="FR_hip_joint"   gear="1"/>
    <motor name="FR_thigh_motor" joint="FR_thigh_joint" gear="1"/>
    <motor name="FR_calf_motor"  joint="FR_calf_joint"  gear="1"/>
    <motor name="FL_hip_motor"   joint="FL_hip_joint"   gear="1"/>
    <motor name="FL_thigh_motor" joint="FL_thigh_joint" gear="1"/>
    <motor name="FL_calf_motor"  joint="FL_calf_joint"  gear="1"/>
    <motor name="RR_hip_motor"   joint="RR_hip_joint"   gear="1"/>
    <motor name="RR_thigh_motor" joint="RR_thigh_joint" gear="1"/>
    <motor name="RR_calf_motor"  joint="RR_calf_joint"  gear="1"/>
    <motor name="RL_hip_motor"   joint="RL_hip_joint"   gear="1"/>
    <motor name="RL_thigh_motor" joint="RL_thigh_joint" gear="1"/>
    <motor name="RL_calf_motor"  joint="RL_calf_joint"  gear="1"/>
  </actuator>

  <sensor>
    <framelinvel name="base_linvel" objtype="site" objname="imu_site"/>
    <frameangvel name="base_angvel" objtype="site" objname="imu_site"/>
    <framequat   name="base_quat"   objtype="site" objname="imu_site"/>
    <accelerometer name="base_accel" site="imu_site"/>
    <gyro          name="base_gyro"  site="imu_site"/>
  </sensor>
</mujoco>
""")
print("Setup complete.")


## 2. Train PPO Policy

Configure training hyperparameters below. Default: **2M steps** with 8 parallel envs.

- Architecture: MLP `[512, 256, 128]` (actor & critic)
- Algorithm: PPO (clip=0.2, lr=3e-4, batch=256)
- Reward: velocity tracking + penalties (no survival bonus)
- Domain randomization: mass ±20%, friction ±50%

Adjust `TOTAL_STEPS` and `N_ENVS` as needed. 2M steps takes roughly 25 min on Colab.


In [ ]:
import sys
import types

# Training configuration
TOTAL_STEPS = 2_000_000  # Adjust as needed (2M ≈ 25 min on Colab)
N_ENVS = 8              # Number of parallel environments

# Run training
sys.path.insert(0, os.getcwd())
from src.training.train import train

args = types.SimpleNamespace(
    total_steps=TOTAL_STEPS,
    n_envs=N_ENVS,
    resume=None,
)
train(args)
print("\nTraining complete!")

## 3. Monitor Training
Launch TensorBoard inline to see reward curves.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/training

## 4. Evaluate Trained Policy

Run the best checkpoint on 20 episodes and print statistics.


In [ ]:
import numpy as np
from stable_baselines3 import PPO
from src.env.cheetah_env import MiniCheetahEnv

# Load best checkpoint
ckpt_path = "checkpoints/best/best_model.zip"
if not os.path.exists(ckpt_path):
    # Fall back to final checkpoint
    ckpt_path = "checkpoints/cheetah_final.zip"

policy = PPO.load(ckpt_path)
env = MiniCheetahEnv(render_mode="none", randomize_domain=False, episode_length=1000)

rewards, lengths = [], []
for ep in range(20):
    obs, _ = env.reset()
    total_r, steps = 0.0, 0
    while True:
        action, _ = policy.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        total_r += reward
        steps += 1
        if done or truncated:
            break
    rewards.append(total_r)
    lengths.append(steps)
    print(f"Episode {ep+1:3d}: reward={total_r:8.2f}, steps={steps}")

env.close()

print(f"\n{'='*50}")
print(f"Mean reward:   {np.mean(rewards):8.2f} ± {np.std(rewards):.2f}")
print(f"Mean length:   {np.mean(lengths):8.1f}")
print(f"Survival rate: {sum(1 for l in lengths if l >= 999)/len(lengths)*100:.0f}%")

## 5. Visualize Policy (Video)

Render the trained policy and display as an inline video.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

policy = PPO.load(ckpt_path)
env = MiniCheetahEnv(render_mode="rgb_array", randomize_domain=False, episode_length=500)

frames = []
obs, _ = env.reset()
for _ in range(500):
    action, _ = policy.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env.step(action)
    frame = env.render()
    if frame is not None:
        frames.append(frame)
    if done or truncated:
        break
env.close()

if frames:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis("off")
    im = ax.imshow(frames[0])

    def update(i):
        im.set_data(frames[i])
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=20, blit=True)
    plt.close(fig)
    HTML(ani.to_jshtml())
else:
    print("No frames rendered — check MuJoCo rgb_array support")

## 6. Reward Component Analysis

Analyze the reward breakdown across a rollout.


In [ ]:
import matplotlib.pyplot as plt

policy = PPO.load(ckpt_path)
env = MiniCheetahEnv(render_mode="none", randomize_domain=False, episode_length=500)

component_history = {k: [] for k in [
    "r_linvel", "r_yaw", "r_orientation", "r_lin_vel_z",
    "r_ang_vel_xy", "r_height", "r_torque", "r_smooth", "r_total"
]}

obs, _ = env.reset()
for _ in range(500):
    action, _ = policy.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env.step(action)
    if "reward_components" in info:
        for k, v in info["reward_components"].items():
            if k in component_history:
                component_history[k].append(v)
    if done or truncated:
        break
env.close()

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle("Reward Components Over Episode", fontsize=14)
for ax, (name, values) in zip(axes.flat, component_history.items()):
    ax.plot(values, linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel("Step")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Print averages
print("\nAverage reward components:")
for name, values in component_history.items():
    if values:
        print(f"  {name:20s}: {np.mean(values):+.4f}")

## 7. Download Trained Model
Download the best checkpoint to your local machine.

In [ ]:
from google.colab import files

if os.path.exists("checkpoints/best/best_model.zip"):
    files.download("checkpoints/best/best_model.zip")
    print("Downloaded best_model.zip")
elif os.path.exists("checkpoints/cheetah_final.zip"):
    files.download("checkpoints/cheetah_final.zip")
    print("Downloaded cheetah_final.zip")
else:
    print("No checkpoint found — train first!")